# **Bibliometria dos anais (edições I até XXI) do SBSR**

---


*Pipeline* automatizado para busca, filtragem temática de artigos (desastres) e consolidação bibliométrica.

-----
### 1. Configuração do ambiente e dependências

In [1]:
!pip install -q selenium pandas beautifulsoup4
!apt-get remove -y chromium-chromedriver > /dev/null 2>&1
!apt-get update -qq && apt-get install -y -qq wget unzip curl > /dev/null 2>&1
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb > /dev/null 2>&1
!apt-get install -f -y -qq > /dev/null 2>&1
!rm -f google-chrome-stable_current_amd64.deb

import re
import subprocess

versao_chrome = subprocess.check_output(["google-chrome", "--version"]).decode()
versao_chrome = re.search(r"(\d+\.\d+\.\d+\.\d+)", versao_chrome).group(1)

url_driver = f"https://storage.googleapis.com/chrome-for-testing-public/{versao_chrome}/linux64/chromedriver-linux64.zip"
!wget -q {url_driver} -O /tmp/chromedriver.zip
!unzip -o /tmp/chromedriver.zip -d /tmp/ > /dev/null
!chmod +x /tmp/chromedriver-linux64/chromedriver
!mv /tmp/chromedriver-linux64/chromedriver /usr/local/bin/chromedriver
!rm -rf /tmp/chromedriver* /tmp/chromedriver.zip

print(f"✔ Chrome e ChromeDriver prontos na versão: {versao_chrome}")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
✔ Chrome e ChromeDriver prontos na versão: 156.0.8078.47


-----
### 2. Definição de parâmetros e regras de busca

In [2]:
SUMARIOS = {
    1978: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/09.25.20.31/doc/@sumario.htm",
    1982: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/10.01.23.18/doc/@sumario.htm",
    1984: "http://marte.sid.inpe.br/col/ltid.inpe.br/sbsr/2009/04.03.17.43/doc/@sumario.htm",
    1986: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2009/03.03.11.29/doc/@sumario.htm",
    1988: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/07.28.17.53/doc/@sumario.htm",
    1990: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/08.26.21.29/doc/@sumario.htm",
    1993: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte@80/2008/06.23.20.41/doc/@sumario.htm",
    1996: "http://marte.sid.inpe.br/col/dpi.inpe.br/lise/2004/08.25.13.20/doc/@sumario.htm",
    1998: "http://marte.sid.inpe.br/col/dpi.inpe.br/lise/2004/08.11.17.13/doc/@sumario.htm",
    2001: "http://marte.sid.inpe.br/col/dpi.inpe.br/lise/2001/12.28.15.53/doc/@sumario.htm",
    2003: "http://marte.sid.inpe.br/col/ltid.inpe.br/sbsr/2003/03.28.12.35/doc/@sumario.htm",
    2005: "http://marte.sid.inpe.br/col/ltid.inpe.br/sbsr/2005/02.12.16.31/doc/@sumario.htm",
    2007: "http://marte.sid.inpe.br/col/dpi.inpe.br/sbsr@80/2007/01.31.19.10/doc/@sumario.htm",
    2009: "http://marte.sid.inpe.br/col/dpi.inpe.br/sbsr@80/2009/03.02.18.19/doc/@sumario.htm",
    2011: "http://marte.sid.inpe.br/col/dpi.inpe.br/marte/2011/09.28.16.42/doc/@sumario.htm",
    2013: "http://marte2.sid.inpe.br/col/dpi.inpe.br/marte2/2013/05.28.22.25/doc/@sumario.htm",
    2015: "http://marte2.sid.inpe.br/col/sid.inpe.br/marte2/2015/05.31.21.54/doc/@sumario.htm",
    2017: "http://marte2.sid.inpe.br/col/sid.inpe.br/marte2/2017/09.25.14.55/doc/@sumario.htm",
    2019: "http://marte2.sid.inpe.br/col/sid.inpe.br/marte2/2019/11.08.12.52/doc/@sumario.htm",
    2023: "http://marte2.sid.inpe.br/col/sid.inpe.br/marte2/2023/05.18.17.15/doc/@sumario.htm",
    2025: "http://marte2.sid.inpe.br/col/sid.inpe.br/marte2/2025/09.17.19.06/doc/@sumario.htm",
}

ROMANOS = {
    1978: "I", 1982: "II", 1984: "III", 1986: "IV", 1988: "V",
    1990: "VI", 1993: "VII", 1996: "VIII", 1998: "IX", 2001: "X",
    2003: "XI", 2005: "XII", 2007: "XIII", 2009: "XIV", 2011: "XV",
    2013: "XVI", 2015: "XVII", 2017: "XVIII", 2019: "XIX",
    2023: "XX", 2025: "XXI"
}

CATEGORIAS = {
    "Deslizamentos": ["deslizamento", "escorregamento", "desmoronamento", "movimento de massa", "movimentos de massa", "landslide"],
    "Inundação": ["inundação", "inundações", "enchente", "alagamento", "enxurrada", "flood", "flooding"],
    "Secas": ["seca", "estiagem", "estiagens", "desertificação", "drought", "desertification"]
}

PAGINAS_IGNORAR = {
    "capa.htm", "pagina_rosto.htm", "ficha_catalografica.htm",
    "prefacio.htm", "comissao.htm", "@indice.htm", "indice.htm",
    "@sumario.htm", "sumario.htm", "pagina_rosto.html",
}

### 3. Funções de *parsing* e normalização

In [3]:
import unicodedata
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def normalizar(texto):
    texto = unicodedata.normalize("NFD", str(texto).lower())
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    texto = re.sub(r"[\-_/\\.,;:!?()\[\]{}'\"`´]+", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()

def construir_padroes():
    padroes = []
    for cat, termos in CATEGORIAS.items():
        for termo in termos:
            base = normalizar(termo)
            padrao = re.compile(rf"\b{re.escape(base)}(s|es)?\b", flags=re.IGNORECASE)
            padroes.append((cat, termo, padrao))
    return padroes

PADROES = construir_padroes()

def eh_linha_institucional(linha):
    linha_lower = linha.lower()
    termos_invalidos = [
        "gerado automaticamente", "urlibservice", "sumario", "sumário",
        "pagina de rosto", "página de rosto", "dados catalograficos",
        "dados catalográficos", "prefacio", "prefácio", "comissoes",
        "comissões", "indice por autor", "índice por autor",
        "exemplo para referenciar", "link para os temas",
        "simpósio brasileiro", "simposio brasileiro", "sbsr", "urlib"
    ]
    return any(p in linha_lower for p in termos_invalidos) or bool(re.match(r"^\d{1,2}\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ]", linha)) or len(linha) < 15

def extrair_artigos_por_links(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    resultados = []
    padrao_autores = re.compile(
        r"([A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+)+"
        r"(?:\s*;\s*[A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+)+)*)"
    )

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        linhas = [l.strip() for l in a.get_text("\n", strip=True).split("\n") if l.strip()]

        if len(linhas) >= 2:
            titulo, autores = linhas[0], "; ".join(linhas[1:])
        else:
            texto_plano = a.get_text(" ", strip=True)
            m = padrao_autores.search(texto_plano)
            if m and m.start() > 25:
                titulo, autores = texto_plano[:m.start()].strip(), m.group(1).strip()
            else:
                titulo, autores = texto_plano, ""

        if href.startswith("#") or href in PAGINAS_IGNORAR:
            continue
        if any(p in href.lower() for p in ["capa", "rosto", "ficha", "prefacio", "comissao", "indice", "sumario", "topo"]):
            continue
        if not re.search(r"\.(pdf|htm|html)$", href, re.IGNORECASE) or len(titulo) < 25 or re.fullmatch(r"[\d\s\.\-]+", titulo):
            continue

        resultados.append({"titulo": titulo, "autores": autores, "link": urljoin(base_url, href)})

    vistos = set()
    return [r for r in resultados if not (r["titulo"] in vistos or vistos.add(r["titulo"]))]

def extrair_artigos_por_texto(html):
    soup = BeautifulSoup(html, "html.parser")
    linhas = [l.strip() for l in soup.get_text("\n", strip=True).split("\n") if l.strip()]
    artigos = []
    i = 0

    while i < len(linhas):
        linha_atual = linhas[i]
        linha_seguinte = linhas[i + 1] if i + 1 < len(linhas) else ""

        if eh_linha_institucional(linha_atual):
            i += 1
            continue

        if ";" in linha_atual and len(linha_atual) > 60 and not re.match(r"^\d{1,2}\s", linha_atual):
            partes = linha_atual.split(";")
            if len(partes[0].strip()) > 25:
                artigos.append({"titulo": partes[0].strip(), "autores": "; ".join(p.strip() for p in partes[1:]), "link": ""})
                i += 1
                continue

        if len(linha_atual) > 25 and ";" not in linha_atual and not re.match(r"^\d{1,2}\s", linha_atual) and not linha_atual.endswith("."):
            if ";" in linha_seguinte or re.search(r"[A-Z][a-z]+,\s*[A-Z]\.", linha_seguinte):
                artigos.append({"titulo": linha_atual, "autores": linha_seguinte, "link": ""})
                i += 2
                continue
        i += 1

    return artigos

def extrair_autores_da_pagina_artigo(html):
    soup = BeautifulSoup(html, "html.parser")
    autores = [m.get("content").strip() for m in soup.find_all("meta", {"name": "citation_author"}) if m.get("content")]

    if not autores:
        texto = soup.get_text(" ", strip=True)
        for p in [r'Autor(?:es)?\s*[:\-]\s*([A-Z][^.]{5,150})', r'By\s+([A-Z][^.]{5,150})']:
            matches = re.findall(p, texto)
            for m in matches:
                autores.extend([n.strip() for n in re.split(r'[;,]', m) if 5 < len(n.strip()) < 80])
            if autores:
                break

    vistos = set()
    limpos = [a for a in autores if a and len(a) > 3 and not (a in vistos or vistos.add(a))]
    return "; ".join(limpos)

def extrair_artigos_com_frames(driver, url):
    try:
        driver.set_page_load_timeout(40)
        driver.get(url)
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(2)

        artigos = extrair_artigos_por_links(driver.page_source, url)
        if len(artigos) < 5:
            candidatos = extrair_artigos_por_texto(driver.page_source)
            if len(candidatos) > len(artigos):
                artigos = candidatos

        if len(artigos) < 5:
            frames = driver.find_elements(By.TAG_NAME, "frame") + driver.find_elements(By.TAG_NAME, "iframe")
            for frame in frames:
                try:
                    driver.switch_to.frame(frame)
                    time.sleep(1.5)
                    cand_links = extrair_artigos_por_links(driver.page_source, driver.current_url)
                    cand_texto = extrair_artigos_por_texto(driver.page_source)
                    driver.switch_to.default_content()
                    melhor = max([artigos, cand_links, cand_texto], key=len)
                    if len(melhor) > len(artigos):
                        artigos = melhor
                except Exception:
                    driver.switch_to.default_content()
        return artigos
    except Exception as e:
        try:
            driver.switch_to.default_content()
        except Exception:
            pass
        return []

-----
### 4. Execução da busca automatizada e mineração dos artigos

In [5]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("--lang=pt-BR")
options.add_argument("user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36")

driver = webdriver.Chrome(service=Service("/usr/local/bin/chromedriver"), options=options)

ocorrencias, falhas = [], []
total_artigos_analisados = 0

print(f"{'Ano':<6} | {'Edição':<8} | {'Artigos totais':<15} | {'Filtrados':<12}")
print("-" * 48)

for ano, url_sumario in SUMARIOS.items():
    artigos = extrair_artigos_com_frames(driver, url_sumario)
    if not artigos:
        falhas.append(ano)
        print(f"{ano:<6} | {ROMANOS.get(ano, str(ano)):<8} | {'FALHA/VAZIO':<15} | {'0':<12}")
        continue

    total_artigos_analisados += len(artigos)
    vistos_na_edicao = set()
    encontrados_edicao = 0

    for item in artigos:
        titulo = item["titulo"]
        autores = item.get("autores", "")
        link = item.get("link", "")

        if not autores and ";" in titulo:
            partes = titulo.split(";")
            if len(partes[0].strip()) > 25:
                autores = "; ".join(p.strip() for p in partes[1:])
                titulo = partes[0].strip()

        t_norm = normalizar(titulo)
        if t_norm in vistos_na_edicao:
            continue
        vistos_na_edicao.add(t_norm)

        matches = [(cat, termo, m.group(0)) for cat, termo, regex in PADROES if (m := regex.search(t_norm))]
        if not matches:
            continue

        if not autores and link:
            try:
                driver.set_page_load_timeout(15)
                driver.get(link)
                WebDriverWait(driver, 8).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
                autores = extrair_autores_da_pagina_artigo(driver.page_source)
            except Exception:
                pass

        por_categoria = {}
        for cat, termo, encontrado in matches:
            por_categoria.setdefault(cat, []).append((termo, encontrado))

        for cat, pares in por_categoria.items():
            ocorrencias.append({
                "Categoria": cat,
                "Palavras-Chave": "; ".join(sorted({t for t, _ in pares})),
                "Evidencia_Titulo": "; ".join(sorted({e for _, e in pares})),
                "Edicao SBSR": ROMANOS.get(ano, str(ano)),
                "Ano": ano,
                "Titulo": titulo,
                "Autores": autores,
            })
            encontrados_edicao += 1

    print(f"{ano:<6} | {ROMANOS.get(ano, str(ano)):<8} | {len(artigos):<15} | {encontrados_edicao:<12}")

driver.quit()

df = pd.DataFrame(ocorrencias).sort_values(["Ano", "Categoria", "Titulo"]).reset_index(drop=True)
caminho_csv = "/content/bibliometria_sbsr_com_autores.csv"
df.to_csv(caminho_csv, index=False, encoding="utf-8-sig", sep=";")

print("-" * 48)
print(f"✔ Coleta finalizada: {len(df)} registros úteis minerados de {total_artigos_analisados} artigos totais.")

Ano    | Edição   | Artigos totais  | Filtrados   
------------------------------------------------
1978   | I        | 84              | 0           
1982   | II       | 107             | 2           
1984   | III      | 47              | 0           
1986   | IV       | 211             | 5           
1988   | V        | 103             | 0           
1990   | VI       | 133             | 0           
1993   | VII      | 23              | 1           
1996   | VIII     | 175             | 0           
1998   | IX       | 152             | 3           
2001   | X        | 261             | 5           
2003   | XI       | 396             | 6           
2005   | XII      | 601             | 11          
2007   | XIII     | 962             | 24          
2009   | XIV      | 1012            | 25          
2011   | XV       | 1207            | 28          
2013   | XVI      | 1206            | 36          
2015   | XVII     | 998             | 22          
2017   | XVIII    | 1111         